# Notebook 22 — Zero-Cost Ensemble Variants (Phase 4 Tier 1)

**Goal:** Evaluate two untried ensemble combos using existing OOF artifacts — no new training required.

| Combo | Models | Status |
|-------|--------|--------|
| **1A — 3-model (no MLP)** | LGBM NB12 + LSTM NB18 + Hybrid NB21 | ⭐ Most interesting — MLP was neutral in NB20 (MLP+Hybrid CV = Hybrid solo = 0.9185) |
| **1B — 2-model neural** | LSTM NB18 + Hybrid NB21 | CV=0.9206 reported in NB21 post-hoc, never submitted |

**Hypothesis:** Removing MLP (ρ=0.9186 vs Hybrid = zero additive gain) from the 4-model ensemble may
recover the LB boost from +0.0141 (4-model) toward +0.0177 (Hybrid solo).

**Reference scores:**
- Hybrid solo (v006): CV 0.9185, LB **0.93618** — current best
- 4-model NB20 (v007): CV 0.9205, LB **0.93463** — worse than solo
- Boost confirmed: neural solo +0.0177, 4-model neural +0.0141

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

# Path detection — works in VSCode (walk-up) and on Kaggle (fixed path)
if Path('/kaggle/input').exists():
    parquets = list(Path('/kaggle/input').rglob('train_features_v2.parquet'))
    project_root = parquets[0].parents[2] if parquets else Path('/kaggle/working')
    processed_dir   = project_root / 'data' / 'processed'
    submissions_dir = Path('/kaggle/working')
else:
    cwd = Path.cwd()
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
    project_root    = cwd
    processed_dir   = project_root / 'data' / 'processed'
    submissions_dir = project_root / 'submissions'

print(f'Project root  : {project_root}')
print(f'Processed dir : {processed_dir.exists()}')
print(f'Submissions   : {submissions_dir.exists()}')

BOOST_SOLO   = 0.0177   # confirmed v006 Hybrid solo
BOOST_4MODEL = 0.0141   # confirmed v007 4-model

## 1 — Load OOF Predictions

In [ ]:
oof12 = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
oof18 = pd.read_parquet(processed_dir / 'oof_predictions_nb18.parquet')
oof21 = pd.read_parquet(processed_dir / 'oof_predictions_nb21.parquet')

# Align on id
oof = oof21[['id', 'fold', 'PitNextLap']].copy()
oof = oof.merge(oof12[['id', 'lgbm_tuned_oof']], on='id')
oof = oof.merge(oof18[['id', 'lstm_oof']],       on='id')
oof = oof.merge(oof21[['id', 'hybrid_oof']],     on='id')

print(f'OOF rows: {len(oof):,}')
print(f'Columns : {list(oof.columns)}')
assert oof.isnull().sum().sum() == 0, 'NaN in OOF!'

y_true = oof['PitNextLap'].values
lgbm_oof   = oof['lgbm_tuned_oof'].values
lstm_oof   = oof['lstm_oof'].values
hybrid_oof = oof['hybrid_oof'].values

print(f'\nSolo AUCs:')
print(f'  LGBM   : {roc_auc_score(y_true, lgbm_oof):.4f}')
print(f'  LSTM   : {roc_auc_score(y_true, lstm_oof):.4f}')
print(f'  Hybrid : {roc_auc_score(y_true, hybrid_oof):.4f}')

## 2 — Spearman ρ Matrix

In [ ]:
models = {
    'LGBM'  : lgbm_oof,
    'LSTM'  : lstm_oof,
    'Hybrid': hybrid_oof,
}

names = list(models.keys())
preds = list(models.values())
n = len(names)

print('Spearman ρ matrix:')
header = '         ' + ''.join(f'{nm:>10}' for nm in names)
print(header)
for i, ni in enumerate(names):
    row = f'{ni:<9}'
    for j, nj in enumerate(names):
        rho, _ = spearmanr(preds[i], preds[j])
        row += f'{rho:>10.4f}'
    print(row)

## 3 — Rank-Average Helper

In [ ]:
def rank_average(arrays):
    """Equal-weight rank average across a list of 1-D arrays."""
    ranks = np.array([rankdata(a) / len(a) for a in arrays])
    return ranks.mean(axis=0)

## 4 — Evaluate All Combos (OOF)

In [ ]:
combos = {
    'Hybrid solo (baseline)'           : [hybrid_oof],
    '1B: LSTM + Hybrid'                : [lstm_oof, hybrid_oof],
    '1A: LGBM + LSTM + Hybrid (no MLP)': [lgbm_oof, lstm_oof, hybrid_oof],
    'LGBM + Hybrid'                    : [lgbm_oof, hybrid_oof],
    '4-model NB20 baseline (MLP ref)'  : None,   # reported from memory
}

results = {}
print(f'{"Combo":<40} {"CV AUC":>8}  {"Delta vs solo":>14}  {"Proj LB (solo boost)":>22}  {"Proj LB (4m boost)":>20}')
print('-' * 115)

solo_auc = roc_auc_score(y_true, hybrid_oof)

for name, arrays in combos.items():
    if arrays is None:
        auc = 0.9205  # from NB20
        note = '(from NB20 memory)'
    else:
        avg = rank_average(arrays)
        auc = roc_auc_score(y_true, avg)
        note = ''
    delta = auc - solo_auc
    proj_solo = auc + BOOST_SOLO
    proj_4m   = auc + BOOST_4MODEL
    results[name] = {'auc': auc, 'delta': delta}
    flag = ' <-- IMPROVEMENT' if delta > 0.001 else ('  <-- noise floor' if 0 < delta <= 0.001 else '')
    print(f'{name:<40} {auc:>8.4f}  {delta:>+14.4f}  {proj_solo:>22.4f}  {proj_4m:>20.4f}  {note}{flag}')

print(f'\nNoise floor: CV gains < +0.001 do not reliably transfer to LB')
print(f'Current best LB: 0.93618 (Hybrid solo, v006)')

## 5 — Per-Fold AUC for Best Combo

In [ ]:
# Report per-fold for 1A and 1B
for combo_name, arrays in [
    ('1A: LGBM + LSTM + Hybrid', [lgbm_oof, lstm_oof, hybrid_oof]),
    ('1B: LSTM + Hybrid',        [lstm_oof, hybrid_oof]),
]:
    avg = rank_average(arrays)
    oof_aug = oof.copy()
    oof_aug['ensemble'] = avg
    fold_aucs = []
    for fold_id in sorted(oof_aug['fold'].unique()):
        mask = oof_aug['fold'] == fold_id
        fauc = roc_auc_score(oof_aug.loc[mask, 'PitNextLap'], oof_aug.loc[mask, 'ensemble'])
        fold_aucs.append(fauc)
    print(f'{combo_name}')
    print(f'  Fold AUCs : {[round(a,4) for a in fold_aucs]}')
    print(f'  Mean      : {np.mean(fold_aucs):.4f}')
    print(f'  Std       : {np.std(fold_aucs):.4f}')
    print()

## 6 — Build Submission CSVs

In [ ]:
test12 = pd.read_parquet(processed_dir / 'test_predictions_nb12.parquet')
test18 = pd.read_parquet(processed_dir / 'test_predictions_nb18.parquet')
test21 = pd.read_parquet(processed_dir / 'test_predictions_nb21.parquet')

test = test21[['id']].copy()
test = test.merge(test12[['id', 'lgbm_tuned_pred']], on='id')
test = test.merge(test18[['id', 'lstm_pred']],       on='id')
test = test.merge(test21[['id', 'hybrid_pred']],     on='id')

print(f'Test rows: {len(test):,}')
assert test.isnull().sum().sum() == 0, 'NaN in test!'

lgbm_pred   = test['lgbm_tuned_pred'].values
lstm_pred   = test['lstm_pred'].values
hybrid_pred = test['hybrid_pred'].values

In [ ]:
# 1A: LGBM + LSTM + Hybrid
test['PitNextLap'] = rank_average([lgbm_pred, lstm_pred, hybrid_pred])
sub_1a = test[['id', 'PitNextLap']].sort_values('id').reset_index(drop=True)

assert sub_1a['id'].duplicated().sum() == 0,       'Duplicate IDs!'
assert sub_1a['PitNextLap'].isnull().sum() == 0,   'NaN predictions!'
assert (sub_1a['PitNextLap'] > 0).all() and (sub_1a['PitNextLap'] <= 1).all(), 'Out of [0,1]!'

out_1a = submissions_dir / 'submission_v008_lgbm_lstm_hybrid.csv'
sub_1a.to_csv(out_1a, index=False)
print(f'Saved 1A: {out_1a}')
print(f'Shape: {sub_1a.shape}, PitNextLap range: [{sub_1a.PitNextLap.min():.4f}, {sub_1a.PitNextLap.max():.4f}]')

In [ ]:
# 1B: LSTM + Hybrid
test['PitNextLap'] = rank_average([lstm_pred, hybrid_pred])
sub_1b = test[['id', 'PitNextLap']].sort_values('id').reset_index(drop=True)

assert sub_1b['id'].duplicated().sum() == 0,       'Duplicate IDs!'
assert sub_1b['PitNextLap'].isnull().sum() == 0,   'NaN predictions!'
assert (sub_1b['PitNextLap'] > 0).all() and (sub_1b['PitNextLap'] <= 1).all(), 'Out of [0,1]!'

out_1b = submissions_dir / 'submission_v009_lstm_hybrid.csv'
sub_1b.to_csv(out_1b, index=False)
print(f'Saved 1B: {out_1b}')
print(f'Shape: {sub_1b.shape}, PitNextLap range: [{sub_1b.PitNextLap.min():.4f}, {sub_1b.PitNextLap.max():.4f}]')

## 7 — Decision Summary

In [ ]:
print('='*70)
print('PHASE 4 TIER 1 — DECISION SUMMARY')
print('='*70)
print(f'Current best LB : 0.93618  (Hybrid solo, v006)')
print()

for name, arrays, version, boost_label, boost in [
    ('1A: LGBM+LSTM+Hybrid (3-model)', [lgbm_oof, lstm_oof, hybrid_oof], 'v008', 'solo boost', BOOST_SOLO),
    ('1B: LSTM+Hybrid (2-model)',       [lstm_oof, hybrid_oof],           'v009', 'solo boost', BOOST_SOLO),
]:
    avg  = rank_average(arrays)
    auc  = roc_auc_score(y_true, avg)
    delta = auc - solo_auc
    proj_lb_lo = auc + BOOST_4MODEL  # pessimistic: boost stays at 4-model rate
    proj_lb_hi = auc + BOOST_SOLO    # optimistic: boost recovers to solo rate
    above_noise = delta > 0.001
    print(f'{name}')
    print(f'  CV AUC    : {auc:.4f}  (delta vs solo: {delta:+.4f})')
    print(f'  Above noise floor (+0.001): {above_noise}')
    print(f'  Proj LB   : {proj_lb_lo:.4f} (pessimistic) — {proj_lb_hi:.4f} (optimistic)')
    print(f'  Submit as : submissions/{version}_*.csv')
    recommend = 'SUBMIT' if above_noise else 'SKIP (below noise floor)'
    print(f'  Verdict   : {recommend}')
    print()